In [5]:
import torch

class YOLOWrapper(torch.nn.Module):
    def __init__(self, yolo_model):
        super().__init__()
        self.model = yolo_model

    def forward(self, x):
        output = self.model(x)
        if isinstance(output, (tuple, list)):
            return output[0]   # take just the main prediction tensor
        return output

In [6]:
from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import cv2
import numpy as np
import torch

def generate_heatmap(image_path, model):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image at: {image_path}")

    # Wrap the underlying torch model
    wrapped_model = YOLOWrapper(model.model)
    wrapped_model.eval()

    target_layers = [model.model.model[-2]]
    cam = EigenCAM(wrapped_model, target_layers)

    img_resized = cv2.resize(img, (640, 640))
    rgb_img = img_resized[:, :, ::-1] / 255.0

    device = next(model.model.parameters()).device
    input_tensor = torch.from_numpy(rgb_img).permute(2, 0, 1).unsqueeze(0).float().to(device)

    grayscale_cam = cam(input_tensor)[0]
    cam_image = show_cam_on_image(rgb_img.astype(np.float32), grayscale_cam, use_rgb=True)
    return cam_image

In [12]:
from ultralytics import YOLO


model = YOLO("runs/detect/yolo11n_gpu_run1/weights/best.pt")

image_path = "data/images/test/AN_unpaved_44.jpg"
img_name='AN_unpaved_44.jpg'
results = model(image_path, conf=0.1)
results[0].save(filename="demo_output/real-world-img/result-AN_unpaved_44.jpg")

heatmap = generate_heatmap(image_path, model)
cv2.imwrite(f"demo_output/real-world-img/heatmap_{img_name}", heatmap)
print(f"Done: {img_name}")


image 1/1 c:\Users\hp410\Desktop\Harshil\Python programs\Road Damage Detection Model\data\images\test\AN_unpaved_44.jpg: 384x640 3 unpaved_roads, 239.7ms
Speed: 4.9ms preprocess, 239.7ms inference, 12.6ms postprocess per image at shape (1, 3, 384, 640)
Done: AN_unpaved_44.jpg


In [ ]:
from model_pipeline import detect_damage
import os
demo_images =  ['UnPavedRoad__44.jpg'] # ADD IMAGES NAMES IN THIS LIST
for img in demo_images:
    path = f'data/images/test/{img}'
    try:
        result = detect_damage(path, output_dir="presentation_assets")
        print(f"✓ {img}: {result['num_detections']} detections")
        print(f"\nImage: {result['image_path']}")
        print(f"Damage found: {result['damage_found']} ({result['num_detections']} detection(s))")
        print(f"Boxed image saved to: {result['boxed_image_path']}")
        print(f"Heatmap saved to: {result['heatmap_image_path']}")
        
        for i, det in enumerate(result["detections"], 1):
            print(f"\n  Detection {i}:")
            print(f"    Class: {det['class']}")
            print(f"    Confidence: {det['confidence']*100:.1f}%")
            print(f"    Severity: {det['severity']} (score: {det['severity_score']})")
            print(f"    Explanation: {det['explanation']}")
    except Exception as e:
        print(f"✗ {img} failed: {e}")
    results = model(path, conf=0.1)
    results[0].save(filename="demo_output/real-world-img/result-AN_unpaved_44.jpg")

    heatmap = generate_heatmap(image_path, model)
    cv2.imwrite(f"demo_output/real-world-img/heatmap_{img_name}", heatmap)
    print(f"Done: {img_name}")

image 1/1 c:\Users\hp410\Desktop\Harshil\Python programs\Road Damage Detection Model\data\images\test\UnPavedRoad__44.jpg: 384x640 1 unpaved_road, 34.9ms
Speed: 1462.0ms preprocess, 34.9ms inference, 4.5ms postprocess per image at shape (1, 3, 384, 640)
✓ UnPavedRoad__44.jpg: 1 detections

Image: data/images/test/UnPavedRoad__44.jpg
Damage found: True (1 detection(s))
Boxed image saved to: presentation_assets\UnPavedRoad__44_boxes.jpg
Heatmap saved to: presentation_assets\UnPavedRoad__44_heatmap.jpg

  Detection 1:
    Class: unpaved_road
    Confidence: 75.2%
    Severity: Critical (score: 0.86)
    Explanation: Detected unpaved road with 75.2% confidence, covering approximately 20.3% of the visible area. This is a stretch of road lacking proper paving, causing dust, instability, and wear. Classified as Critical severity based on size, damage type, and model confidence.
